# Getting Started with OPERA DIST-ALERT-HLS Product
---

This notebook serves as an introduction to the OPERA Land Surface Disturbance Alert from Harmonized Landsat and Sentinel-2 (`DIST-ALERT-HLS`) product. The `DIST-HLS` product suite captures vegetation disturbance applicable to a variety of disasters including wildfire, flooding, and landsliding. The `DIST-ALERT-HLS` product explored here captures vegetation disturbance due to wildfires in the greater Los Angeles, California area in January 2025. 

To explore use-case applications of the OPERA `DIST-ALERT-HLS` product, see our example notebooks on [wildfire impacts](https://github.com/OPERA-Cal-Val/OPERA_Applications/blob/main/DIST/DIST_ALERT/Wildfire/McKinney.ipynb) and [flooding and landslides](https://github.com/OPERA-Cal-Val/OPERA_Applications/blob/main/DIST/DIST_ALERT/Flood_and_Landslide/NorthCarolina_Helene_Flood_Landslides_Sept2024.ipynb).

*<font color='red'>Note: Please refer to [DIST product specification](https://lpdaac.usgs.gov/documents/1766/OPERA_DIST_HLS_Product_Specification_V1.pdf) for more information. </font>*<br>

### The OPERA `DIST-HLS` Product Suite
---
The OPERA Land Surface Disturbance from Harmonized Landsat and Sentinel-2 (`DIST-HLS`) product suite maps vegetation disturbance using surface reflectance data from Harmonized Landsat and Sentinel-2 (`HLS`). Disturbance is identified when vegetation cover declines or spectral variation falls outside the historical norm for a given `HLS` pixel.

The suite includes two complementary products:

- **`DIST-ALERT-HLS`**: Detects vegetation disturbance at the native `HLS` cadence (every 2–3 days).
- **`DIST-ANN-HLS`**: Summarizes confirmed disturbances from `DIST-ALERT-HLS` over the previous calendar year.

`DIST-ALERT-HLS` is distributed as 19 GeoTIFF layers plus a metadata file, organized in folders corresponding to each input `HLS` tile.  
`DIST-ANN-HLS` includes 21 GeoTIFF layers and a metadata file per tile.

`DIST-HLS` data are projected onto the Military Grid Reference System (MGRS). Each tile spans 109.8 km², consisting of 3,660 x 3,660 pixels at 30-meter resolution, with ~4.9 km of overlap with neighboring tiles.
Detailed descriptions of the raster layers and their properties are available in the [OPERA DIST-HLS Product Specification Document](https://lpdaac.usgs.gov/documents/1766/OPERA_DIST_HLS_Product_Specification_V1.pdf).

### HLS Data 
---
The Harmonized Landsat and Sentinel-2 (`HLS`) dataset provides surface reflectance (SR) data from two satellite sensors:

- **OLI** (Operational Land Imager) on Landsat-8  
- **MSI** (Multi-Spectral Instrument) on Sentinel-2A and 2B

`HLS` data are projected onto the Military Grid Reference System (MGRS). Each tile spans 109.8 km², consisting of 3,660 x 3,660 pixels at 30-meter resolution, with ~4.9 km of overlap with neighboring tiles.

### Accessing the Data
---
`DIST-HLS` products are publicly available via NASA’s Distributed Active Archive Centers (DAACs), specifically through the [LP DAAC](https://lpdaac.usgs.gov).

## Library Imports
First we import the necessary Python libraries. These come pre-installed in the `opera_app` anaconda environement within the [OPERA Applications Gitub repository](https://github.com/OPERA-Cal-Val/OPERA_Applications). We also import a collection of custom DIST-specific functions from a source file called `dist_utils.py`.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

import earthaccess
import xarray as xr
import hvplot.xarray
import geoviews as gv
from pyproj import Proj
import holoviews as hv

from bokeh.models import FixedTicker
hv.extension('bokeh')

import sys
sys.path.append('../../../')
from src.dist_utils import *

## Authentication with NASA Earthdata credentials
A [NASA Earthdata Login](https://urs.earthdata.nasa.gov/) account is required to download the data used in this tutorial. You can create an account at the link provided. After establishing an account, the code in the next cell will verify authentication. If this is your first time running the notebook, you will be prompted to enter your Earthdata login credentials, which will be saved in `~/.netrc`.

In [ ]:
auth = earthaccess.login(strategy="netrc")
s3_credentials = auth.get_s3_credentials(daac="PODAAC")

The next cell configures the `gdal` library and provideds necessary authentication to successfully access cloud-hosted assets.

In [ ]:
# Check for valid Earthdata credentials
auth = earthaccess.login(strategy="netrc")
s3_credentials = auth.get_s3_credentials(daac="PODAAC")

# Set GDAL configs to successfully access Cloud Assets via vsicurl
gdal.SetConfigOption("GDAL_HTTP_UNSAFESSL", "YES")
gdal.SetConfigOption('GDAL_HTTP_COOKIEFILE','~/cookies.txt')
gdal.SetConfigOption('GDAL_HTTP_COOKIEJAR', '~/cookies.txt')
gdal.SetConfigOption('GDAL_DISABLE_READDIR_ON_OPEN','FALSE')
gdal.SetConfigOption('CPL_VSIL_CURL_ALLOWED_EXTENSIONS','TIF')

## Access and load OPERA `DIST-ALERT-HLS` product layers
The two cells below access and load the OPERA `DIST-ALERT-HLS` data layers into a geocube for visualization.

In [ ]:
product = 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/OPERA_L3_DIST-ALERT-HLS_V1/OPERA_L3_DIST-ALERT-HLS_T11SLT_20250130T182826Z_20250201T033615Z_L9_30_v1/OPERA_L3_DIST-ALERT-HLS_T11SLT_20250130T182826Z_20250201T033615Z_L9_30_v1_'
layer_names = ['VEG-ANOM-MAX', 'VEG-DIST-DATE', 'VEG-DIST-STATUS']
layer_paths = [f"{product}{layer}.tif" for layer in layer_names]

In [ ]:
# Create geocube of stacked bands
da, crs = stack_layers(layer_paths)

# Create basemap
base = gv.tile_sources.EsriImagery.opts(width=1000, height=1000, padding=0.1)

## Visualize the layers of the OPERA `DIST-ALERT-HLS` product

### **Band 1: Maximum Vegetation Anomaly Value (VEG_ANOM_MAX)**
***

**Data Type:** UInt8<br>
**Description:** Difference between historical and current year observed vegetation cover at the date of maximum decrease, measured on scale from 0-100%<br>

In [ ]:
veg_anom_max = da.z.where(da['z'] != 255).sel({'layer': 1})
veg_anom_max = veg_anom_max.where(veg_anom_max != 0)

veg_anom_max.hvplot.image(
    x='longitude',
    y='latitude',
    crs=crs,
    dynamic=True,
    aspect='equal',
    frame_width=500,
    frame_height=500,
    cmap='hot',
    clabel='Vegetation Loss (%)',
    clim=(0, 100),
    alpha=0.8
).opts(
    title="DIST-ALERT-HLS; VEG-ANOM-MAX",
    xlabel='Longitude',
    ylabel='Latitude'
).redim.nodata(value=255) * base

### **Band 2: Date of Initial Vegetation Disturbance (VEG_DIST_DATE)**
***

**Data Type:** Int16<br>
**Description:** Day of first loss anomaly detection in the last year, denoted as the number of days since December 31st, 2020.<br>

In [ ]:
veg_dist_date = da.z.where(da['z'] != -1).sel({'layer': 2})
veg_dist_date = veg_dist_date.where(veg_dist_date != 0)

veg_dist_date.hvplot.image(
    x='longitude',
    y='latitude',
    crs=crs,
    dynamic=True,
    aspect='equal',
    frame_width=500,
    frame_height=500,
    cmap='inferno',
    clabel='Days since 12/31/2020',
    alpha=0.8,
    clim=(0, 592)
).opts(
    title="DIST-ALERT-HLS; VEG-DIST-DATE",
    xlabel='Longitude',
    ylabel='Latitude'
) * base

### **Band 3: Vegetation Disturbance Status (VEG_DIST_STATUS)**
***

**Data Type:** UInt8<br>
**Description:** Indication of vegetation cover loss (vegetation disturbance); "provisional" is used from the second detection until vegetation disturbance is detected for consecutive number of HLS scenes, when it is then labeled "confirmed."<br>

In [ ]:
color_key = {
    1: "#005555",
    2: "#897f4e",
    3: "#dee043",
    4: "#008888",
    5: "#e48727",
    6: "#e01b07",
    7: "#777777",
    8: "#dddddd",
}

labels_txt = {
    1: "1: First detection, <50%",
    2: "2: Provisional, <50%",
    3: "3: Confirmed, <50%",
    4: "4: First detection, ≥50%",
    5: "5: Provisional, ≥50%%",
    6: "6: Confirmed, ≥50%",
    7: "7: Confirmed, <50%, completed",
    8: "8: Confirmed, ≥50%, completed",
}

levels = sorted(color_key)
code_to_index   = {c: i for i, c in enumerate(levels)}
index_to_color  = [color_key[c] for c in levels]
index_to_label  = {i: labels_txt[c] for i, c in enumerate(levels)}

veg_dist_status = da.z.where(da['z'] != 255).sel({'layer': 3})
veg_idx         = veg_dist_status.where(veg_dist_status != 0).copy()
veg_idx.data    = np.vectorize(code_to_index.get)(veg_idx.data)

N = len(index_to_color)
ticks  = list(range(N))
ticker = FixedTicker(ticks=ticks)
clim   = (-0.5, N - 0.5)

(veg_idx.hvplot.image(
        x='longitude',
        y='latitude',
        crs=crs,
        frame_width=500,
        frame_height=500,
        aspect='equal',
        cmap=index_to_color,
        clim=clim,
        alpha=0.8)
 .opts(title="DIST-ALERT-HLS; VEG-DIST-STATUS",
       xlabel='Longitude', ylabel='Latitude',
       colorbar_opts={'ticker': ticker,
                      'major_label_overrides': index_to_label})
 * base)

**Layer Values:**<br> 
* **0:** No disturbance<br>
* **1:** First disturbance detection with vegetation cover change <50% <br>
* **2:** Provisional (**two consecutive disturbance detections**) with vegetation cover change <50% <br>
* **3:** Confirmed (**recurrent detection**) Disturbance with vegetation cover change <50% <br> 
* **4:** First disturbance detection with vegetation cover change ≥50% <br>
* **5:** Provisional (**two consecutive disturbance detections**) with vegetation cover change ≥50% <br>
* **6:** Confirmed (**recurrent detection**) Disturbance with vegetation cover change ≥50% <br> 
* **7:** Confirmed (**recurrent detection**) Disturbance with vegetation cover change <50%, completed <br> 
* **8:** Confirmed (**recurrent detection**) Disturbance with vegetation cover change ≥50%, completed <br> 
* **255:** No data

### Conclusion
This has been an overview of how to access, load, and visualize three layers of the OPERA `DIST-ALERT-HLS` product applied to vegetation change related to wildfire. For more detail about the layers not visualized here, please see the [DIST product specification](https://lpdaac.usgs.gov/documents/1766/OPERA_DIST_HLS_Product_Specification_V1.pdf). For additional use-case example Jupyter Notebooks, see the [OPERA Applications](https://github.com/OPERA-Cal-Val/OPERA_Applications) GitHub repository.